In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# Load data
df = pd.read_csv('/content/synthetic_wind_energy_dataset_30k.csv')

# Clean 'power' column
df['power'] = df['power'].str.replace(' kWh', '').astype(float)

# Convert timestamp and extract features
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24)
df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24)

# Define features/target
features = ['wind_speed', 'wind_dir', 'temp', 'pressure', 'RH', 'gusts', 'dew_pt', 'precip', 'sin_hour', 'cos_hour', 'terrain']
target = 'power'

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(df[features], df[target], test_size=0.2, random_state=42)

# Numeric and categorical
numeric_features = ['wind_speed', 'wind_dir', 'temp', 'pressure', 'RH', 'gusts', 'dew_pt', 'precip', 'sin_hour', 'cos_hour']
categorical_features = ['terrain']

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

### RANDOM FOREST GRID SEARCH
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [8, 10, 12],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1)
rf_grid.fit(X_train, y_train)

rf_best = rf_grid.best_estimator_
rf_pred = rf_best.predict(X_val)
rf_r2 = r2_score(y_val, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))
rf_mae = mean_absolute_error(y_val, rf_pred)

print(f'\nBest Random Forest Params: {rf_grid.best_params_}')
print(f'Random Forest R²: {rf_r2:.4f}, RMSE: {rf_rmse:.4f}, MAE: {rf_mae:.4f}')

### XGBOOST GRID SEARCH
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(objective='reg:squarederror', random_state=42))
])

xgb_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [6, 8, 10],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.8, 1.0]
}

xgb_grid = GridSearchCV(xgb_pipeline, xgb_param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1)
xgb_grid.fit(X_train, y_train)

xgb_best = xgb_grid.best_estimator_
xgb_pred = xgb_best.predict(X_val)
xgb_r2 = r2_score(y_val, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_pred))
xgb_mae = mean_absolute_error(y_val, xgb_pred)

print(f'\nBest XGBoost Params: {xgb_grid.best_params_}')
print(f'XGBoost R²: {xgb_r2:.4f}, RMSE: {xgb_rmse:.4f}, MAE: {xgb_mae:.4f}')


Fitting 3 folds for each of 24 candidates, totalling 72 fits

Best Random Forest Params: {'model__max_depth': 12, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Random Forest R²: 0.9986, RMSE: 6.1160, MAE: 2.0927
Fitting 3 folds for each of 36 candidates, totalling 108 fits

Best XGBoost Params: {'model__learning_rate': 0.05, 'model__max_depth': 6, 'model__n_estimators': 200, 'model__subsample': 0.8}
XGBoost R²: 0.9987, RMSE: 5.7126, MAE: 1.8100


In [2]:
import pickle

# Example: Saving the XGBoost best model (you can also save rf_best)
with open('best_xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb_best, f)

print("✅ Model saved as 'best_xgb_model.pkl'")


✅ Model saved as 'best_xgb_model.pkl'


In [4]:
import pickle
import numpy as np
import pandas as pd

# Load the saved model
with open('/content/best_xgb_model.pkl', 'rb') as f:
    model = pickle.load(f)

# Define the input columns in order
input_columns = ['wind_speed', 'wind_dir', 'temp', 'pressure', 'RH', 'gusts', 'dew_pt', 'precip', 'hour', 'terrain']

# Collect manual input
wind_speed = float(input("Enter wind speed: "))
wind_dir = float(input("Enter wind direction: "))
temp = float(input("Enter temperature: "))
pressure = float(input("Enter pressure: "))
RH = float(input("Enter relative humidity (RH): "))
gusts = float(input("Enter gust speed: "))
dew_pt = float(input("Enter dew point: "))
precip = float(input("Enter precipitation: "))
hour = int(input("Enter hour (0-23): "))
terrain = input("Enter terrain type (forest/open/urban): ")

# Transform hour to sin and cos
sin_hour = np.sin(2 * np.pi * hour / 24)
cos_hour = np.cos(2 * np.pi * hour / 24)

# Prepare input DataFrame
input_data = pd.DataFrame([{
    'wind_speed': wind_speed,
    'wind_dir': wind_dir,
    'temp': temp,
    'pressure': pressure,
    'RH': RH,
    'gusts': gusts,
    'dew_pt': dew_pt,
    'precip': precip,
    'sin_hour': sin_hour,
    'cos_hour': cos_hour,
    'terrain': terrain
}])

# Predict
predicted_power = model.predict(input_data)

print(f"⚡ Predicted Power Output: {predicted_power[0]:.2f} kWh")


Enter wind speed: 11.41
Enter wind direction: 305
Enter temperature: 29.8
Enter pressure: 1031.2
Enter relative humidity (RH): 47
Enter gust speed: 7.22
Enter dew point: 12.5
Enter precipitation: 1.15
Enter hour (0-23): 2
Enter terrain type (forest/open/urban): forest
⚡ Predicted Power Output: 438.12 kWh


In [10]:
import requests
import pandas as pd
import numpy as np
import pickle
from datetime import datetime, timedelta

# Load your trained model
with open('/content/best_xgb_model.pkl', 'rb') as f:
    model = pickle.load(f)

LAT, LON = 22.7592, 78.7535  # Gadarwara NTPC
TERRAIN = 'open'

# Fetch 5-day hourly forecast
params = {
    'latitude': LAT,
    'longitude': LON,
    'hourly': 'temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,wind_gusts_10m,precipitation',
    'forecast_days': 6
}
response = requests.get('https://api.open-meteo.com/v1/forecast', params=params)
data = response.json()

# Build DataFrame
hourly = data['hourly']
df = pd.DataFrame(hourly)
df['time'] = pd.to_datetime(df['time'])

# Get tomorrow's date
today = datetime.utcnow().date()
tomorrow = today + timedelta(days=1)

# Filter data starting from tomorrow (00:00)
df_tomorrow = df[df['time'].dt.date >= tomorrow].reset_index(drop=True)

input_rows = []
for i in range(len(df_tomorrow)):
    wind_speed = df_tomorrow['wind_speed_10m'][i]
    wind_dir = df_tomorrow['wind_direction_10m'][i]
    temp = df_tomorrow['temperature_2m'][i]
    pressure = 1013  # Assume standard pressure or pull from another source
    RH = df_tomorrow['relative_humidity_2m'][i]
    gusts = df_tomorrow['wind_gusts_10m'][i]
    dew_pt = temp - ((100 - RH) / 5.)  # simple dew point estimate
    precip = df_tomorrow['precipitation'][i]
    timestamp = df_tomorrow['time'][i]
    hour = timestamp.hour
    sin_hour = np.sin(2 * np.pi * hour / 24)
    cos_hour = np.cos(2 * np.pi * hour / 24)

    row = {
        'wind_speed': wind_speed,
        'wind_dir': wind_dir,
        'temp': temp,
        'pressure': pressure,
        'RH': RH,
        'gusts': gusts,
        'dew_pt': dew_pt,
        'precip': precip,
        'sin_hour': sin_hour,
        'cos_hour': cos_hour,
        'terrain': TERRAIN
    }
    input_rows.append(row)

# Predict
input_df = pd.DataFrame(input_rows)
predictions = model.predict(input_df)
total_predicted_power = np.sum(predictions)

# Results
print("\n⚡ 5-Day Wind Energy Forecast (starting tomorrow, hourly):")
for i, (dt, pred) in enumerate(zip(df_tomorrow['time'], predictions)):
    print(f"{dt}: {pred:.2f} kWh")

print(f"\n🔋 Total 5-Day Predicted Wind Energy Production (from tomorrow): {total_predicted_power:.2f} kWh")



⚡ 5-Day Wind Energy Forecast (starting tomorrow, hourly):
2025-05-02 00:00:00: 8.57 kWh
2025-05-02 01:00:00: 2.72 kWh
2025-05-02 02:00:00: 32.42 kWh
2025-05-02 03:00:00: 105.07 kWh
2025-05-02 04:00:00: 33.28 kWh
2025-05-02 05:00:00: 1.99 kWh
2025-05-02 06:00:00: 148.79 kWh
2025-05-02 07:00:00: 57.66 kWh
2025-05-02 08:00:00: 18.89 kWh
2025-05-02 09:00:00: 40.11 kWh
2025-05-02 10:00:00: 91.90 kWh
2025-05-02 11:00:00: 1235.67 kWh
2025-05-02 12:00:00: 1072.95 kWh
2025-05-02 13:00:00: 1272.69 kWh
2025-05-02 14:00:00: 281.73 kWh
2025-05-02 15:00:00: 230.80 kWh
2025-05-02 16:00:00: 145.22 kWh
2025-05-02 17:00:00: 35.73 kWh
2025-05-02 18:00:00: 8.00 kWh
2025-05-02 19:00:00: 0.38 kWh
2025-05-02 20:00:00: 1.38 kWh
2025-05-02 21:00:00: 2.34 kWh
2025-05-02 22:00:00: -0.54 kWh
2025-05-02 23:00:00: -0.93 kWh
2025-05-03 00:00:00: -0.76 kWh
2025-05-03 01:00:00: 2.87 kWh
2025-05-03 02:00:00: 0.23 kWh
2025-05-03 03:00:00: -0.11 kWh
2025-05-03 04:00:00: 3.81 kWh
2025-05-03 05:00:00: 72.23 kWh
2025-05-03